# Mini-Project 1: Weather Forecast with Python Web APIs

**Student:** Tumuhirwe Iden · **Student number:** S26M25/011 · MSCS & MSDS OOP with Python, Assignment 2 (Advent 2026)

This notebook takes a place name, finds its coordinates, gets a daily forecast, and shows the forecast as a summary, a table and a chart. It uses these approved APIs:

| Provider | Purpose |
|---|---|
| Open-Meteo Geocoding (`geocoding-api.open-meteo.com/v1/search`) | Place name to latitude and longitude |
| Open-Meteo Forecast (`api.open-meteo.com/v1/forecast`) | Main forecast |
| MET Norway Locationforecast 2.0 (`api.met.no/weatherapi/locationforecast/2.0`) | Second source (Part C) |

**Data sources:** Weather data by [Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0). Location data by Open-Meteo / GeoNames (CC BY 4.0). Forecast data from [MET Norway](https://api.met.no/) (CC BY 4.0 and NLOD 2.0).

The reusable code is in `src/weather/`. This notebook imports it.

**Libraries.** The notebook uses only the allowed libraries (`urllib`, `requests`, `json`, `datetime`, `time`, `os`, `pandas`, `matplotlib`) and these others:
- `sys` and `pathlib`: to add `src/` to the import path.
- `dataclasses`, `typing`, `collections.abc`: for typed data classes and type hints (standard library, no runtime effect).
- `unittest.mock`: to simulate "no network" in Part C without a real network failure (standard library).
- `pytest` (tests only): to run the offline checks in `tests/`.

## Configuration

This is the **only cell that you must edit**. Put your university email in the `WEATHER_CONTACT_EMAIL` environment variable, or replace the placeholder below. Do not commit a real email to a shared repository. The `.example` placeholder is not a real address.

In [1]:
import os

LOCATION = "Kampala"          # Place name to search for
LOCATION_CHOICE = 1           # Number of the candidate to use if the name is ambiguous (1 = first)
FORECAST_DAYS = 7             # 1 to 16 days
CONTACT_EMAIL = os.environ.get("WEATHER_CONTACT_EMAIL", "CHANGE-ME@university.example")
INTERACTIVE = False           # True: ask with input() when the name is ambiguous

## Setup

In [2]:
import json
import sys
from collections.abc import Callable
from pathlib import Path
from unittest.mock import Mock

import pandas as pd
import requests
from IPython.display import display

sys.path.insert(0, str(Path.cwd() / "src"))

from weather.config import build_user_agent, GEOCODING_URL, OPEN_METEO_FORECAST_URL
from weather.display import format_forecast_table, plot_forecast, plot_source_comparison, summarise_forecast
from weather.exceptions import ApiRequestError, WeatherError
from weather.forecast import forecast_to_dataframe, get_forecast
from weather.geocoding import choose_location, geocode, parse_geocoding_response
from weather.http_basics import geocode_with_requests, geocode_with_urllib
from weather.http_client import JsonHttpClient, RetryPolicy
from weather.met_norway import compare_sources, get_met_forecast, met_to_daily_dataframe

USER_AGENT = build_user_agent(CONTACT_EMAIL)
client = JsonHttpClient(USER_AGENT, timeout=10.0)
pd.set_option("display.width", 120)
print("User-Agent is set. Contact email from:",
      "environment variable" if os.environ.get("WEATHER_CONTACT_EMAIL") else "placeholder (please set WEATHER_CONTACT_EMAIL)")

User-Agent is set. Contact email from: environment variable


## Part A: HTTP fundamentals

### A.1 Geocoding with `urllib` only

The function `geocode_with_urllib` in `src/weather/http_basics.py` uses only `urllib.request`, `urllib.parse` and `json`. It builds the query string with `urllib.parse.urlencode`.

In [3]:
urllib_info = geocode_with_urllib("Dresden", USER_AGENT)
print(urllib_info.url)
first = urllib_info.body["results"][0]
print(first["name"], first["country"], first["latitude"], first["longitude"])

https://geocoding-api.open-meteo.com/v1/search?name=Dresden&count=5&language=en&format=json
Dresden Germany 51.05089 13.73832


### A.2 The same call with `requests`

`geocode_with_requests` uses the `params=` argument and a timeout.

In [4]:
requests_info = geocode_with_requests("Dresden", USER_AGENT)
print(requests_info.url)
print(requests_info.body["results"][0] == urllib_info.body["results"][0])

https://geocoding-api.open-meteo.com/v1/search?name=Dresden&count=5&language=en&format=json
True


### A.3 Comparison of the two approaches

| Point | `urllib` (standard library) | `requests` |
|---|---|---|
| **Code length** | Longer. You build the URL with `urlencode`, make a `Request` object, open it in a `with` block, read bytes, decode them, and call `json.loads`. | Shorter. One `requests.get(url, params=..., timeout=...)` call and `response.json()`. |
| **Error handling** | `urlopen` raises `HTTPError` for 4xx and 5xx codes and `URLError` for network errors. To read the error body, you must read the exception object. | No error for a bad status by default. You call `raise_for_status()`. Exceptions have a clear tree: `ConnectionError`, `Timeout`, `HTTPError`, all under `RequestException`. |
| **Headers** | Pass a `headers` dict to `Request`. The default User-Agent is `Python-urllib/3.x`. | Pass `headers=`. A `Session` keeps headers for all calls. The default User-Agent is `python-requests/x.y`. |
| **Encoding** | You get bytes and must decode them yourself (`.decode("utf-8")`). | `response.json()` and `response.text` find the encoding for you. You can set `response.encoding` to change it. |
| **Extra features** | None. No sessions, no connection reuse, no `elapsed` time. | Sessions with connection reuse, `response.elapsed`, retries with adapters, and easy timeouts. |

**Conclusion.** `urllib` needs no install, so it is good for small scripts. `requests` gives shorter and clearer code with better error types, so the rest of this project uses `requests`.

### A.4 Status code, Content-Type and response time of one request

In [5]:
pd.DataFrame(
    {
        "urllib": [urllib_info.status_code, urllib_info.content_type, round(urllib_info.elapsed_ms, 1)],
        "requests": [requests_info.status_code, requests_info.content_type, round(requests_info.elapsed_ms, 1)],
    },
    index=["Status code", "Content-Type", "Response time (ms)"],
)

,urllib,requests
Status code,200,200
Content-Type,application/json; charset=utf-8,application/json; charset=utf-8
Response time (ms),701.1,694.7


Both calls return status **200** and `application/json`. The `requests` time comes from `response.elapsed` (time until the headers arrive). The `urllib` time is measured with `time.perf_counter()` and includes reading the body. So the two times are not exactly the same measure. The times change from run to run.

## Part B: Forecast retrieval and display

### B.1 `geocode(name)`: candidates, zero results and ambiguous names

`geocode` returns a list of `Location` objects. When nothing matches, the API leaves out the `results` key, so `geocode` returns an empty list. `choose_location` then raises `LocationNotFoundError`.

An ambiguous name such as "Springfield" gives many candidates. The notebook lists them, and the user picks one with a number. To keep "Restart & Run All" working, the number comes from the configuration cell. If `INTERACTIVE = True`, the notebook asks with `input()`.

In [6]:
springfields = geocode("Springfield", client=client)
for number, place in enumerate(springfields, start=1):
    print(f"{number:>2}. {place.label}")
print("Picked:", choose_location("Springfield", springfields, choice=3).label)

 1. Springfield, Missouri, United States (37.2153, -93.2982)
 2. Springfield, Illinois, United States (39.8017, -89.6437)
 3. Springfield, Massachusetts, United States (42.1015, -72.5898)
 4. Springfield, Ohio, United States (39.9242, -83.8088)
 5. Springfield, Tennessee, United States (36.5092, -86.8850)
 6. Palmyra, Missouri, United States (39.7942, -91.5232)
 7. Jackson, Minnesota, United States (43.6208, -94.9886)
 8. Springfield, Kentucky, United States (37.6853, -85.2222)
 9. Springfield, Georgia, United States (32.3724, -81.3115)
10. Springfield, Colorado, United States (37.4083, -102.6144)
Picked: Springfield, Massachusetts, United States (42.1015, -72.5898)


In [7]:
print("Results for an unknown name:", geocode("Qwxzzyplort", client=client))

Results for an unknown name: []


In [8]:
candidates = geocode(LOCATION, client=client)
location = choose_location(
    LOCATION, candidates,
    choice=None if INTERACTIVE else LOCATION_CHOICE,
    input_func=input if INTERACTIVE else None,
)
print(f"{len(candidates)} candidates for {LOCATION!r}. Using: {location.label}, time zone {location.timezone}")

10 candidates for 'Kampala'. Using: Kampala, Central Region, Uganda (0.3163, 32.5822), time zone Africa/Kampala


### B.2 `get_forecast(lat, lon, days)`

The request asks for the daily maximum and minimum temperature, precipitation sum, maximum wind speed, and the WMO weather code. `timezone=auto` makes the dates match the local time of the location.

In [9]:
payload = get_forecast(location.latitude, location.longitude, FORECAST_DAYS, client=client)
print("Time zone:", payload["timezone"], "| Units:", payload["daily_units"])

Time zone: Africa/Kampala | Units: {'time': 'iso8601', 'weather_code': 'wmo code', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h'}


### B.3 Convert the JSON to a pandas DataFrame

`forecast_to_dataframe` puts the unit in each column name. It also checks that the units in the response are the units in the column names. The weather code is decoded with the WMO table from the Open-Meteo documentation (`src/weather/weather_codes.py`).

In [10]:
forecast = forecast_to_dataframe(payload)
forecast.head(3)

,Weather,Max temperature (°C),Min temperature (°C),Precipitation (mm),Max wind speed (km/h),WMO code
Date,,,,,,
2026-10-05,Light drizzle,27.3,18.6,1.6,14.2,51
2026-10-06,Moderate rain showers,27.9,18.8,5.2,17.4,81
2026-10-07,Moderate rain showers,26.4,19.0,9.2,10.9,81


### B.4 Display: text summary, formatted table and chart

In [11]:
print(summarise_forecast(forecast, location))

Forecast for Kampala, Central Region, Uganda (0.3163, 32.5822), 05 Oct to 11 Oct 2026 (time zone Africa/Kampala).
- Warmest day: Tue 06 Oct, up to 27.9 °C.
- Coolest night: Sun 11 Oct, down to 18.3 °C.
- Total precipitation: 61.1 mm; 7 of 7 days have 1 mm or more. Wettest day: Sun 11 Oct (29.5 mm).
- Strongest wind: 17.4 km/h.
- Most common weather: moderate rain showers.
Source: Weather data by Open-Meteo.com (CC BY 4.0).


In [12]:
format_forecast_table(forecast, location)

,Weather,Max temperature (°C),Min temperature (°C),Precipitation (mm),Max wind speed (km/h)
Date,,,,,
Mon 05 Oct,Light drizzle,27.3,18.6,1.6,14.2
Tue 06 Oct,Moderate rain showers,27.9,18.8,5.2,17.4
Wed 07 Oct,Moderate rain showers,26.4,19.0,9.2,10.9
Thu 08 Oct,Moderate drizzle,27.2,18.4,1.6,10.6
Fri 09 Oct,Thunderstorm (slight or moderate),26.9,18.4,8.0,12.5
Sat 10 Oct,Moderate drizzle,25.4,18.4,6.0,13.8
Sun 11 Oct,Moderate rain showers,23.8,18.3,29.5,8.5


In [13]:
plot_forecast(forecast, location)

<Figure size 1000x600 with 2 Axes>

## Part C: Robustness and a second source

### C.1 One function for every request

Every API call goes through `JsonHttpClient.get_json` (`src/weather/http_client.py`). It:

- sets a **timeout** on each attempt;
- calls **`raise_for_status()`**;
- **retries** connection errors, timeouts, 5xx and 429 responses with **exponential backoff** (0.5 s, 1 s, 2 s, ..., up to 8 s) and a **retry cap** (`max_attempts`, 4 by default);
- does **not** retry other 4xx errors, because the same request gives the same error;
- raises the custom **`ApiRequestError`** on final failure. The error keeps the URL, the number of attempts, the status code and the API's reason text;
- keeps successful responses in a small in-memory **cache**, so a repeated call does not send traffic again.

### Second source: MET Norway

MET Norway needs a User-Agent with an app name and a contact. It sends hourly and 6-hourly steps in UTC. `met_to_daily_dataframe` changes them to daily values in the local time zone. We compare only the days that MET covers fully (24 hours).

In [14]:
met_payload = get_met_forecast(location.latitude, location.longitude, client=client)
met_daily = met_to_daily_dataframe(met_payload, location.timezone or forecast.attrs["timezone"])
comparison = compare_sources(forecast, met_daily)
comparison.round(1)

,Max temperature (°C) Open-Meteo,Min temperature (°C) Open-Meteo,Precipitation (mm) Open-Meteo,Max temperature (°C) MET Norway,Min temperature (°C) MET Norway,Precipitation (mm) MET Norway,Max temperature (°C) difference,Min temperature (°C) difference,Precipitation (mm) difference
Date,,,,,,,,,
2026-10-06,27.9,18.8,5.2,28.1,18.8,8.1,-0.2,0.0,-2.9
2026-10-07,26.4,19.0,9.2,26.3,19.1,7.0,0.1,-0.1,2.2
2026-10-08,27.2,18.4,1.6,27.1,19.0,2.4,0.1,-0.6,-0.8
2026-10-09,26.9,18.4,8.0,25.8,19.8,10.6,1.1,-1.4,-2.6
2026-10-10,25.4,18.4,6.0,24.4,19.3,13.3,1.0,-0.9,-7.3
2026-10-11,23.8,18.3,29.5,23.8,18.5,18.9,0.0,-0.2,10.6


In [15]:
plot_source_comparison(comparison, location)

<Figure size 1000x450 with 1 Axes>

In [16]:
mean_abs = comparison.filter(like="difference").abs().mean().round(1)
print("Mean absolute difference, Open-Meteo minus MET Norway:")
print(mean_abs.to_string())

Mean absolute difference, Open-Meteo minus MET Norway:
Max temperature (°C) difference    0.4
Min temperature (°C) difference    0.5
Precipitation (mm) difference      4.4


The two models agree on the general pattern. Small differences in temperature are normal: the models use different grids, and the MET daily values come from instant hourly or 6-hourly temperatures, so they can miss the true peak. Precipitation differs more, because rain showers are hard to forecast at one point.

### C.2 Behaviour with broken inputs

Each case below catches the error and prints it, so "Run All" continues. The `on_retry` function prints each retry. For the demonstrations, the waits are short.

In [17]:
def show_retry(attempt: int, delay: float, reason: str) -> None:
    print(f"  attempt {attempt} failed ({reason}); waiting {delay:.2f} s before a retry")


def demo(title: str, action: Callable[[], object]) -> None:
    print(f"--- {title}")
    try:
        result = action()
        print("  No error:", result)
    except WeatherError as error:
        print(f"  {type(error).__name__}: {error}")


fast_retry = RetryPolicy(max_attempts=3, backoff_base=0.2, max_backoff=1.0)

In [18]:
demo("1. Invalid city name",
     lambda: choose_location("Qwxzzyplort", geocode("Qwxzzyplort", client=client)))

--- 1. Invalid city name
  LocationNotFoundError: No place matches 'Qwxzzyplort'.


In [19]:
demo("2a. Invalid latitude, checked before the request",
     lambda: get_forecast(999.0, 32.58, client=client))

raw_client = JsonHttpClient(USER_AGENT, retry=fast_retry, on_retry=show_retry)
demo("2b. Invalid latitude sent to the API (HTTP 400 is not retried)",
     lambda: raw_client.get_json(OPEN_METEO_FORECAST_URL,
                                 {"latitude": 999, "longitude": 32.58, "daily": "temperature_2m_max"}))

--- 2a. Invalid latitude, checked before the request
  InvalidCoordinatesError: Latitude 999.0 must be from -90 to 90.
--- 2b. Invalid latitude sent to the API (HTTP 400 is not retried)


  ApiRequestError: The API returned an error (url=https://api.open-meteo.com/v1/forecast, attempts=1, status=400, reason='Latitude must be in range of -90 to 90°. Given: 999.0.')


In [20]:
offline_session = Mock(spec=requests.Session)
offline_session.get.side_effect = requests.ConnectionError("Simulated: network is unreachable")
offline_client = JsonHttpClient(USER_AGENT, session=offline_session, retry=fast_retry,
                                on_retry=show_retry, use_cache=False)
demo("3. No network (simulated)", lambda: geocode("Dresden", client=offline_client))
print("  Calls made:", offline_session.get.call_count)

--- 3. No network (simulated)
  attempt 1 failed (ConnectionError); waiting 0.20 s before a retry


  attempt 2 failed (ConnectionError); waiting 0.40 s before a retry


  ApiRequestError: Request failed: ConnectionError (url=https://geocoding-api.open-meteo.com/v1/search, attempts=3)
  Calls made: 3


In [21]:
tiny_timeout_client = JsonHttpClient(USER_AGENT, timeout=0.001, retry=fast_retry,
                                     on_retry=show_retry, use_cache=False)
demo("4. Tiny timeout (0.001 s)", lambda: geocode("Dresden", client=tiny_timeout_client))

--- 4. Tiny timeout (0.001 s)
  attempt 1 failed (ConnectTimeout); waiting 0.20 s before a retry


  attempt 2 failed (ConnectTimeout); waiting 0.40 s before a retry


  ApiRequestError: Request failed: ConnectTimeout (url=https://geocoding-api.open-meteo.com/v1/search, attempts=3)


**Results.** The invalid city gives `LocationNotFoundError`. The invalid latitude is stopped by local validation (`InvalidCoordinatesError`). If it is sent, the API returns HTTP 400 with a clear reason, and the client does not retry. No network and a tiny timeout are both retried with growing waits, and then end with `ApiRequestError` after the retry cap.

### Offline checks

The full test suite is in `tests/` (run `pytest`). The checks below use the saved JSON in `tests/data/`, so they run without a network.

In [22]:
def load(name: str) -> dict[str, object]:
    return json.loads(Path("tests/data", name).read_text(encoding="utf-8"))

assert parse_geocoding_response(load("geocode_no_results.json")) == []
assert len(parse_geocoding_response(load("geocode_springfield.json"))) > 1
assert parse_geocoding_response(load("geocode_dresden.json"))[0].country == "Germany"
sample = forecast_to_dataframe(load("forecast_kampala.json"))
assert (sample["Max temperature (°C)"] >= sample["Min temperature (°C)"]).all()
assert sample["Weather"].iloc[0] == "Light drizzle"
assert (met_to_daily_dataframe(load("met_kampala.json"), "Africa/Kampala")["Precipitation (mm)"] >= 0).all()
print("All offline checks passed.")

All offline checks passed.


## Part D: Report

### 1. Data flow from user input to the displayed forecast

The program has a simple pipeline. Each step is one function in `src/weather/`, and each step gives a clear result to the next step.

```
 Configuration cell
 (LOCATION, LOCATION_CHOICE, CONTACT_EMAIL)
        │
        ▼
 geocode(name) ──HTTPS GET──► Open-Meteo Geocoding API
        │  list[Location]            (JSON: results[])
        ▼
 choose_location(...)  ── no match ──► LocationNotFoundError
        │  one Location     ── many matches, no choice ──► AmbiguousLocationError
        ▼
 get_forecast(lat, lon, days) ──HTTPS GET──► Open-Meteo Forecast API
        │  JSON (daily arrays)         get_met_forecast ──► MET Norway API
        ▼                                     │ JSON (UTC time series)
 forecast_to_dataframe(...)                   ▼
        │  DataFrame (units in names)   met_to_daily_dataframe(...)
        ▼                                     │
 summary text · formatted table · chart ◄─────┴── compare_sources(...)
```

All HTTP calls go through one object, `JsonHttpClient`. It adds the User-Agent, the timeout, the retries and the cache. So the rules for network use are in one place, and the other functions only work with data.

First, the user writes a place name in the configuration cell. The geocoding API returns candidate places with coordinates and a time zone. The program selects one candidate. Next, the program asks Open-Meteo for a daily forecast for those coordinates. The JSON has one list for each variable. The program changes these lists into a pandas DataFrame, decodes the weather codes, and checks the units. Last, the program makes a text summary, a table and a chart from the DataFrame. For Part C, the program also gets the MET Norway forecast. It changes the MET time series into daily values in the same local time zone and compares the two sources.

### 2. Licensing and usage terms

**Open-Meteo (forecast and geocoding).** The free API needs no key. It is for **non-commercial use only**. The fair-use limits at the time of writing are less than 10,000 calls per day, 5,000 per hour and 600 per minute. The data is under **CC BY 4.0**, so each output must give credit to Open-Meteo. The geocoding data comes from GeoNames, which is also under CC BY 4.0. A commercial product must buy an API subscription. The subscription uses a different server and an API key, and it has a service agreement.

**MET Norway Locationforecast 2.0.** The data is free, also for commercial use. It is under **CC BY 4.0** and the Norwegian **NLOD 2.0** licence, so credit to MET Norway is necessary. The terms of service have strict technical rules:
- every request must have a User-Agent that identifies the app and gives a contact (an email or a web site). Requests without one can get **403 Forbidden**;
- clients must cache the data, obey the `Expires` header and use `If-Modified-Since`, and must not send more requests than necessary;
- coordinates should have at most 4 decimals;
- there is a total limit of about 20 requests per second for each application. Heavy users must make an agreement with MET, or they get **429** responses or a block.

**What this means for a real product.** A free hobby app can use both APIs if it shows the credits. A commercial app cannot use the free Open-Meteo API. It must pay for Open-Meteo or use MET Norway. The app must show the credit lines on screen. In this notebook, the credits are in the summary text, the table caption and the charts. A product with many users must not call the APIs directly from each phone. A server should collect the requests, cache the forecasts for some minutes, and send one request for each place. This keeps the traffic in the limits and makes the app faster. The contact email in the User-Agent must belong to the team that runs the product, so MET can contact them if there is a problem. The email must not be written in the code. In this project, it comes from an environment variable.

### 3. Reliability issues and how the code handles them

| Issue that we met | How the code handles it |
|---|---|
| The geocoding response has **no `results` key** when nothing matches. | `parse_geocoding_response` uses `payload.get("results", [])`. `choose_location` then raises `LocationNotFoundError` with a clear message. |
| **Ambiguous names** ("Springfield" gives 10 places in the US). | `choose_location` lists the candidates. The user picks one with a number from the configuration cell, or with `input()`. With no choice, it raises `AmbiguousLocationError`. It never guesses. |
| **Slow or missing network**, and **server errors** (5xx, 429). | Each attempt has a timeout. The client retries with exponential backoff (0.5 s, 1 s, 2 s ...) and a cap on attempts. Then it raises `ApiRequestError` with the URL, the number of attempts and the cause. |
| **Bad input**, for example a latitude of 999. The API returns HTTP 400 with a JSON `reason`. | The forecast functions check the ranges before the request. If a 400 still comes back, the client does not retry it, and it puts the API's reason in the error. |
| **MET Norway refuses requests** without a correct User-Agent. | `build_user_agent` always adds the app name and a contact email, and it checks that the email has a valid form. |
| **Different time steps and time zones.** MET sends UTC times, with hourly steps first and 6-hour steps later. | `met_to_daily_dataframe` changes UTC to the local time zone. It uses the 1-hour rain when it exists, and the 6-hour rain otherwise, so rain is not counted two times. The comparison uses only days with 24 hours of data. |
| **Unexpected units or missing fields** in the forecast JSON. | `forecast_to_dataframe` checks each field, the list lengths and the units. If a check fails, it raises `ForecastParseError`. It does not show wrong numbers. |
| **Unknown weather codes.** | `describe_weather_code` returns "Unknown code (N)" and does not fail. |
| **Rate limits.** | A small cache stops repeated calls for the same URL in one session. |

The offline tests in `tests/` use saved JSON files and a mock session. So the parsing and the retry logic can be tested without a network, and the results are the same each time.

### 4. Privacy implications

A weather query looks harmless, but it tells the API provider **where the user is or where the user plans to go**. A place name or a pair of coordinates, together with the time and the **IP address** of the request, can show a home address, a work place, or a travel plan. If the same app sends many queries over time, the provider can build a profile of the movements of a person. Both providers say that they do not sell this data. But the data still goes to a third party, it can be stored in server logs, and it can travel to other countries. Open-Meteo has servers in Europe, and MET Norway is in Norway.

For a real product, the law also applies. In Uganda, the **Data Protection and Privacy Act, 2019** says that location data is personal data when it is linked to a person. In Europe, the GDPR applies. The product must tell users which data goes to which provider, and why.

The design can reduce the risk:
- **Send less detail.** Round the coordinates before the request. MET asks for 4 decimals (about 11 m). For weather, 2 decimals (about 1 km) is enough, and it hides the exact address.
- **Use a server between the users and the APIs.** Then the providers see the server's IP address, not the user's address. The server cache also means that many users share one request.
- **Do not send user identifiers.** The User-Agent has the developer's contact email. It must never have the user's name or email.
- **Use HTTPS only.** Then other people on the network cannot read the queries. All URLs in this project use HTTPS.
- **Keep nothing that is not necessary.** This notebook keeps responses only in memory for one session, and it saves no user location to disk. The sample JSON files in `tests/data/` are for public places only.

To sum up, the APIs are easy to use, but a real product must follow their licence terms, handle network errors, and protect the location data of its users.